[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/OpenCampus_Demo/blob/main/OC_HandwritingOCR.ipynb)


# 手書き文字認識デモ（YomiToku）

日本語に特化した Document AI「[YomiToku](https://github.com/kotaro-kinoshita/yomitoku)」を使い，手書きメモやノートの文字を自動で読み取ります．

**実行環境**: Google Colab（ランタイム → GPU: T4 推奨）

## セルの進め方
1. **設定**（Webカメラの左右反転フラグ）
2. **ライブラリのインストール**
3. **ライブラリの読み込み・モデル準備・サンプル作成**
4. **Gradio の起動**

> インストール直後にエラーが出る場合は，**ランタイム → セッションを再起動**してから「設定」セルとセル3以降を再実行してください．


## 0. 設定（Webカメラ・可視化）

- カメラ映像が左右反転して見える場合は，次のセルの `MIRROR_WEBCAM` を切り替えてください（`True` = ミラー，`False` = 反転なし）．
- 可視化画像上の読み取り文字が小さい／大きい場合は `VIS_FONT_SIZE` を調整してください．
- 変更後は **初期化セル**（モデル準備）と **Gradio 起動セル**を再実行してください．


In [ ]:
# Webカメラの左右反転（ミラー表示）
# True  : 左右反転する（Gradio のデフォルトに近い自撮り表示）
# False : 左右反転しない（紙の文字の向きをそのまま保ちたいとき）
# 文字が左右逆に見える / 見えない場合はここを切り替えてください
MIRROR_WEBCAM = True

# OCR 可視化画像に重ねる読み取り文字の大きさ（YomiToku 既定は 18）
# デモで見やすいよう大きめに設定。変更後は初期化セルと Gradio 起動セルを再実行
VIS_FONT_SIZE = 48

print(f"MIRROR_WEBCAM = {MIRROR_WEBCAM}")
print(f"VIS_FONT_SIZE = {VIS_FONT_SIZE}")


## 1. ライブラリのインストール


In [ ]:
# YomiToku（日本語 Document AI / 手書きOCR）をインストール
# Gradio / OpenCV / Pillow は Colab 標準で利用可能
!pip install -q yomitoku

## 2. ライブラリの読み込み，変数のインスタンス化

モデルのダウンロードと初期化を行います（初回は数分かかることがあります）．


In [ ]:
from __future__ import annotations

import os
import urllib.request
from pathlib import Path

import cv2
import gradio as gr
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from yomitoku import DocumentAnalyzer

# ------------------------------------------------------------
# 定数・パス
# ------------------------------------------------------------
SAMPLE_DIR = Path("samples_handwriting")
FONT_DIR = Path("fonts")
FONT_PATH = FONT_DIR / "ZenKurenaido-Regular.ttf"
FONT_URL = (
    "https://github.com/google/fonts/raw/main/ofl/zenkurenaido/"
    "ZenKurenaido-Regular.ttf"
)

# 手書き風サンプルの文言（高校生向けデモ用）
SAMPLE_TEXTS = [
    ("memo_ai.txt", "今日のオープンキャンパス\nAIで手書き文字を読むデモ"),
    ("memo_schedule.txt", "午後2時から実験室見学\n集合場所は正面玄関"),
    ("memo_note.txt", "機械学習とは\nデータから規則を学ぶ技術"),
]


def resolve_device() -> str:
    """利用可能な推論デバイスを返す．

    Returns:
        str: "cuda" または "cpu"
    """
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU: {name} ({mem_gb:.1f} GB)")
        return "cuda"
    print("GPU が見つかりません．CPU で実行します（時間がかかります）．")
    return "cpu"


def download_font(url: str, save_path: Path) -> Path:
    """手書き風フォントをダウンロードする．

    Args:
        url (str): フォントの URL
        save_path (Path): 保存先パス

    Returns:
        Path: ダウンロードしたフォントのパス
    """
    save_path.parent.mkdir(parents=True, exist_ok=True)
    if save_path.exists():
        return save_path
    print(f"フォントをダウンロード中: {save_path.name}")
    urllib.request.urlretrieve(url, save_path)
    return save_path


def create_handwriting_image(
    text: str,
    font_path: Path,
    image_size: tuple[int, int] = (900, 520),
    font_size: int = 48,
) -> np.ndarray:
    """手書き風の日本語メモ画像を生成する．

    Args:
        text (str): 画像に描画する文字列（改行可）
        font_path (Path): TrueType フォントのパス
        image_size (tuple[int, int]): (幅, 高さ)
        font_size (int): フォントサイズ

    Returns:
        np.ndarray: RGB 画像，形状 (H, W, 3)，dtype=uint8
    """
    width, height = image_size
    # 紙っぽい背景（わずかに暖色）
    img = Image.new("RGB", (width, height), color=(252, 248, 240))
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(str(font_path), font_size)

    # ノートの罫線
    for y in range(90, height - 40, 64):
        draw.line([(48, y), (width - 48, y)], fill=(210, 220, 235), width=2)

    y = 70
    for line in text.split("\n"):
        draw.text((64, y), line, font=font, fill=(35, 45, 70))
        y += 72

    return np.array(img)


def prepare_sample_images(font_path: Path, out_dir: Path) -> list[str]:
    """デモ用サンプル画像を作成し，パス一覧を返す．

    Args:
        font_path (Path): 描画に使うフォントのパス
        out_dir (Path): サンプル画像の出力ディレクトリ

    Returns:
        list[str]: 作成した PNG ファイルパスのリスト
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    paths: list[str] = []
    for filename, text in tqdm(SAMPLE_TEXTS, desc="サンプル作成", leave=False):
        stem = Path(filename).stem
        save_path = out_dir / f"{stem}.png"
        rgb = create_handwriting_image(text, font_path)
        Image.fromarray(rgb).save(save_path)
        paths.append(str(save_path))
    return paths


def rgb_to_bgr(image: np.ndarray) -> np.ndarray:
    """RGB 画像を OpenCV / YomiToku 用の BGR に変換する．

    Args:
        image (np.ndarray): RGB 画像，形状 (H, W, 3)

    Returns:
        np.ndarray: BGR 画像，形状 (H, W, 3)
    """
    return cv2.cvtColor(image, cv2.COLOR_RGB2BGR)


def bgr_to_rgb(image: np.ndarray | None) -> np.ndarray | None:
    """BGR 画像を Gradio 表示用の RGB に変換する．

    Args:
        image (np.ndarray | None): BGR 画像，形状 (H, W, 3)

    Returns:
        np.ndarray | None: RGB 画像，形状 (H, W, 3)．入力が None の場合は None
    """
    if image is None:
        return None
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def resize_short_side(image_bgr: np.ndarray, short_side: int = 720) -> np.ndarray:
    """短辺が short_side ピクセルになるよう縦横比を保ってリサイズする．

    カメラ撮影画像など入力解像度がまちまちでも，OCR 向けに短辺 720px に揃える．

    Args:
        image_bgr (np.ndarray): BGR 画像，形状 (H, W, 3)
        short_side (int): リサイズ後の短辺ピクセル数（既定 720）

    Returns:
        np.ndarray: 短辺が short_side の BGR 画像，形状 (H', W', 3)
    """
    h, w = image_bgr.shape[:2]
    short = min(h, w)
    if short == short_side:
        return image_bgr
    scale = short_side / short
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    # 拡大は CUBIC，縮小は AREA がぼけにくい
    interpolation = cv2.INTER_CUBIC if scale > 1.0 else cv2.INTER_AREA
    return cv2.resize(image_bgr, (new_w, new_h), interpolation=interpolation)


def results_to_text(results) -> str:
    """DocumentAnalyzer の結果から読み取り順のテキストを組み立てる．

    Args:
        results: DocumentAnalyzerSchema（段落・単語情報を含む）

    Returns:
        str: 文字起こし結果の文字列
    """
    if results.paragraphs:
        paragraphs = sorted(
            results.paragraphs,
            key=lambda p: p.order if p.order is not None else 10**9,
        )
        lines = [p.contents for p in paragraphs if p.contents]
        if lines:
            return "\n".join(lines)

    if results.words:
        return "\n".join(w.content for w in results.words if w.content)

    return "（文字を検出できませんでした．明るい場所で大きく写してください．）"


def recognize_handwriting(image: np.ndarray | None) -> tuple[str, np.ndarray | None]:
    """手書き画像を文字起こしし，可視化画像も返す．

    Args:
        image (np.ndarray | None): Gradio から渡される RGB 画像，形状 (H, W, 3)

    Returns:
        tuple[str, np.ndarray | None]:
            - 文字起こしテキスト
            - OCR 可視化画像（RGB），形状 (H, W, 3)
    """
    if image is None:
        return "画像をアップロードするか，カメラで撮影してください．", None

    image_bgr = resize_short_side(rgb_to_bgr(image.astype(np.uint8)), short_side=720)
    results, ocr_vis, _layout_vis = analyzer(image_bgr)
    text = results_to_text(results)
    return text, bgr_to_rgb(ocr_vis)


# ------------------------------------------------------------
# 初期化
# ------------------------------------------------------------
DEVICE = resolve_device()
download_font(FONT_URL, FONT_PATH)
SAMPLE_PATHS = prepare_sample_images(FONT_PATH, SAMPLE_DIR)
print("サンプル画像:", SAMPLE_PATHS)

print("YomiToku DocumentAnalyzer を初期化しています...")
# 可視化の文字サイズを YAML 経由で上書き（text_recognizer.visualize.font_size）
VIS_CFG_PATH = Path("yomitoku_vis.yaml")
VIS_CFG_PATH.write_text(
    f"visualize:\n  font_size: {int(VIS_FONT_SIZE)}\n",
    encoding="utf-8",
)
analyzer_configs = {
    "ocr": {
        "text_recognizer": {
            "path_cfg": str(VIS_CFG_PATH),
        },
    },
}
analyzer = DocumentAnalyzer(
    configs=analyzer_configs,
    visualize=True,
    device=DEVICE,
)
print(f"可視化フォントサイズ: {VIS_FONT_SIZE}")
print("準備完了．次のセルで Gradio を起動してください．")


## 3. Gradio の実行

- **サンプル**タブの例をクリックすると，用意した手書きメモを読み取れます
- **カメラ**でノートやメモを撮影して文字起こしできます
- ファイルのドラッグ＆ドロップにも対応しています
- カメラが左右反転する場合は，上部の `MIRROR_WEBCAM` を変更してこのセルを再実行してください


In [ ]:
def build_demo() -> gr.Blocks:
    """手書き文字認識用の Gradio UI を構築する．

    Returns:
        gr.Blocks: Gradio アプリケーション
    """
    examples = [[path] for path in SAMPLE_PATHS]

    with gr.Blocks(title="手書き文字認識（YomiToku）") as demo:
        gr.Markdown(
            """
            # 手書き文字認識デモ
            **YomiToku** が，手書きの日本語を読み取ります．  
            サンプルを選ぶか，カメラ／アップロードで画像を渡して「文字起こし」を押してください．
            """
        )

        with gr.Row():
            with gr.Column():
                input_image = gr.Image(
                    label="入力画像（アップロード / カメラ）",
                    type="numpy",
                    sources=["upload", "webcam", "clipboard"],
                    height=420,
                    webcam_options=gr.WebcamOptions(mirror=MIRROR_WEBCAM),
                )
                run_btn = gr.Button("文字起こし", variant="primary")
            with gr.Column():
                output_text = gr.Textbox(
                    label="読み取り結果",
                    lines=8,
                    interactive=False,
                )
                output_vis = gr.Image(
                    label="検出結果の可視化（文字位置）",
                    type="numpy",
                    height=420,
                )

        gr.Examples(
            examples=examples,
            inputs=input_image,
            outputs=[output_text, output_vis],
            fn=recognize_handwriting,
            cache_examples=False,
            label="サンプル手書きメモ（クリックで選択＆文字起こし）",
            examples_per_page=3,
        )

        run_btn.click(
            fn=recognize_handwriting,
            inputs=input_image,
            outputs=[output_text, output_vis],
        )

        gr.Markdown(
            """
            ### うまく読むコツ
            - 入力画像は前処理で短辺 720px にリサイズされます
            - 影やピンボケを避け，紙全体が画面に入るようにする
            - 縦書き・くずし字は難易度が上がります
            """
        )

    return demo


demo = build_demo()
# Colab では share=True で外部公開 URL も発行されます
demo.launch(share=True, debug=True)
